In [ ]:
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

In [ ]:
import pandas as pd
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("thedevastator/better-recipes-for-a-better-life")

# List files in the downloaded dataset directory to identify the main data file
for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

# Construct the full path to the recipes.csv file
recipes_file_path = os.path.join(path, 'recipes.csv')

# Load the CSV into a pandas DataFrame
df = pd.read_csv(recipes_file_path)

# Display the first 5 rows of the DataFrame
df.info()

In [ ]:
TAXONOMY = {
    "meal_type":[
        "breakfast",
        "brunch",
        "lunch",
        "dinner",
        "snack",
        "dessert",
        "beverage",
        "appetizer",
        "side_dish"
    ],

    "taste_profile":[
        "sweet",
        "savory",
        "spicy",
        "sour",
        "bitter",
        "umami",
        "fresh"
    ],

    "served_temperature":[
        "cold",
        "warm",
        "hot",
        "room_temperature"
    ],

    "season":[
        "spring",
        "summer",
        "autumn",
        "winter",
        "all_seasons"
    ],

    "difficulty":[
        "easy",
        "medium",
        "hard"
    ]
}

In [ ]:
def build_prompt(recipe):

    return f"""
You are a professional chef.

Classify the following recipe.

Recipe:

Name:
{recipe.recipe_name}

Ingredients:
{recipe.ingredients}

Directions:
{recipe.directions}

Return ONLY valid JSON.

Allowed values

meal_type:
{TAXONOMY["meal_type"]}

taste_profile:
{TAXONOMY["taste_profile"]}

served_temperature:
{TAXONOMY["served_temperature"]}

season:
{TAXONOMY["season"]}

difficulty:
{TAXONOMY["difficulty"]}

JSON schema

{{
    "meal_type":"",
    "taste_profile":[],
    "served_temperature":"",
    "season":[],
    "difficulty":"",
    "main_ingredients":[],
    "characteristics":[],
    "semantic_summary":""
}}
"""

In [ ]:
def classify_recipe(recipe):

    prompt = build_prompt(recipe)

    messages = [
        {
            "role":"user",
            "content":prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=350,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
import json

rows = []

for _, recipe in df.iterrows():

    print("processing recipe: " + recipe.recipe_name)

    response = classify_recipe(recipe)

    try:

        response = (
            response
            .replace("```json","")
            .replace("```","")
            .strip()
        )

        enriched = json.loads(response)

    except Exception:

        print(recipe.recipe_name)

        continue

    enriched["recipe_id"] = recipe["Unnamed: 0"]

    enriched["recipe_name"] = recipe.recipe_name

    rows.append(enriched)

    print(len(rows))

    print(response)

In [ ]:
enriched_df = pd.DataFrame(rows)

In [ ]:
columns_to_clean = ['taste_profile', 'season', 'main_ingredients', 'characteristics']

for col in columns_to_clean:
    if col in enriched_df.columns:
        # Convert list elements to comma-separated strings
        # Use map(str, x) to ensure all elements are strings before joining
        enriched_df[col] = enriched_df[col].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)

print("Characters '[' and ']' removed from relevant columns in enriched_df.")
# Display the first few rows to show the changes
enriched_df.head()

In [ ]:
enriched_df.to_csv(
    "enriched_recipes.csv",
    index=False
)